# Chapter 13 Loading and Preprocessing Data with TensorFlow

## 13.1 The Data API

In [1]:
import tensorflow as tf

x = tf.random.uniform(shape=[4, 3], minval=-1, maxval=1, seed=42)
x

2022-10-21 16:39:38.179794: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-21 16:39:38.250975: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-21 16:39:38.251398: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:936] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-21 16:39:38.252497: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

<tf.Tensor: shape=(4, 3), dtype=float32, numpy=
array([[ 0.9045429 ,  0.35481548,  0.5906365 ],
       [ 0.51156354, -0.04808879,  0.26202965],
       [-0.62795925, -0.7713845 , -0.32755637],
       [ 0.44667006, -0.5616007 ,  0.7146752 ]], dtype=float32)>

In [2]:
dataset = tf.data.Dataset.from_tensor_slices(x)
dataset

<TensorSliceDataset element_spec=TensorSpec(shape=(3,), dtype=tf.float32, name=None)>

In [3]:
for instance in dataset:
    print(instance)

tf.Tensor([0.9045429  0.35481548 0.5906365 ], shape=(3,), dtype=float32)
tf.Tensor([ 0.51156354 -0.04808879  0.26202965], shape=(3,), dtype=float32)
tf.Tensor([-0.62795925 -0.7713845  -0.32755637], shape=(3,), dtype=float32)
tf.Tensor([ 0.44667006 -0.5616007   0.7146752 ], shape=(3,), dtype=float32)


### 13.1.1 Chaining Transformations

In [4]:
for instance in dataset.repeat(3):
    print(instance)

tf.Tensor([0.9045429  0.35481548 0.5906365 ], shape=(3,), dtype=float32)
tf.Tensor([ 0.51156354 -0.04808879  0.26202965], shape=(3,), dtype=float32)
tf.Tensor([-0.62795925 -0.7713845  -0.32755637], shape=(3,), dtype=float32)
tf.Tensor([ 0.44667006 -0.5616007   0.7146752 ], shape=(3,), dtype=float32)
tf.Tensor([0.9045429  0.35481548 0.5906365 ], shape=(3,), dtype=float32)
tf.Tensor([ 0.51156354 -0.04808879  0.26202965], shape=(3,), dtype=float32)
tf.Tensor([-0.62795925 -0.7713845  -0.32755637], shape=(3,), dtype=float32)
tf.Tensor([ 0.44667006 -0.5616007   0.7146752 ], shape=(3,), dtype=float32)
tf.Tensor([0.9045429  0.35481548 0.5906365 ], shape=(3,), dtype=float32)
tf.Tensor([ 0.51156354 -0.04808879  0.26202965], shape=(3,), dtype=float32)
tf.Tensor([-0.62795925 -0.7713845  -0.32755637], shape=(3,), dtype=float32)
tf.Tensor([ 0.44667006 -0.5616007   0.7146752 ], shape=(3,), dtype=float32)


In [5]:
for instance in dataset.repeat(3).batch(5):
    print(instance)

tf.Tensor(
[[ 0.9045429   0.35481548  0.5906365 ]
 [ 0.51156354 -0.04808879  0.26202965]
 [-0.62795925 -0.7713845  -0.32755637]
 [ 0.44667006 -0.5616007   0.7146752 ]
 [ 0.9045429   0.35481548  0.5906365 ]], shape=(5, 3), dtype=float32)
tf.Tensor(
[[ 0.51156354 -0.04808879  0.26202965]
 [-0.62795925 -0.7713845  -0.32755637]
 [ 0.44667006 -0.5616007   0.7146752 ]
 [ 0.9045429   0.35481548  0.5906365 ]
 [ 0.51156354 -0.04808879  0.26202965]], shape=(5, 3), dtype=float32)
tf.Tensor(
[[-0.62795925 -0.7713845  -0.32755637]
 [ 0.44667006 -0.5616007   0.7146752 ]], shape=(2, 3), dtype=float32)


In [6]:
for instance in dataset.repeat(3).batch(5, drop_remainder=True):
    print(instance)

tf.Tensor(
[[ 0.9045429   0.35481548  0.5906365 ]
 [ 0.51156354 -0.04808879  0.26202965]
 [-0.62795925 -0.7713845  -0.32755637]
 [ 0.44667006 -0.5616007   0.7146752 ]
 [ 0.9045429   0.35481548  0.5906365 ]], shape=(5, 3), dtype=float32)
tf.Tensor(
[[ 0.51156354 -0.04808879  0.26202965]
 [-0.62795925 -0.7713845  -0.32755637]
 [ 0.44667006 -0.5616007   0.7146752 ]
 [ 0.9045429   0.35481548  0.5906365 ]
 [ 0.51156354 -0.04808879  0.26202965]], shape=(5, 3), dtype=float32)


In [7]:
dataset = dataset.repeat(3).batch(5)
dataset

<BatchDataset element_spec=TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)>

##### Map

In [8]:
@tf.function
def tf_round(x: tf.Tensor, valid_digits: int = 2) -> tf.Tensor:
    multiplier = 10 ** valid_digits
    return tf.round(x * multiplier, name="round_to_digit") / multiplier

dataset = dataset.map(tf_round, num_parallel_calls=2)
for instance in dataset:
    print(instance)

tf.Tensor(
[[ 0.9   0.35  0.59]
 [ 0.51 -0.05  0.26]
 [-0.63 -0.77 -0.33]
 [ 0.45 -0.56  0.71]
 [ 0.9   0.35  0.59]], shape=(5, 3), dtype=float32)
tf.Tensor(
[[ 0.51 -0.05  0.26]
 [-0.63 -0.77 -0.33]
 [ 0.45 -0.56  0.71]
 [ 0.9   0.35  0.59]
 [ 0.51 -0.05  0.26]], shape=(5, 3), dtype=float32)
tf.Tensor(
[[-0.63 -0.77 -0.33]
 [ 0.45 -0.56  0.71]], shape=(2, 3), dtype=float32)


##### Apply

In [9]:
dataset = dataset.apply(tf.data.Dataset.unbatch)
for instance in dataset:
    print(instance)

tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)


##### Filter

In [10]:
for instance in dataset.filter(lambda instance: tf.reduce_all(tf.greater(tf.abs(instance), 0.2))):
    print(instance)

tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)


##### Take

In [11]:
print(dataset.take(5))
for item in dataset.take(5):
    print(item)

<TakeDataset element_spec=TensorSpec(shape=(3,), dtype=tf.float32, name=None)>
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)


### 13.1.2 Shuffling the Data

In [12]:
dataset = dataset.shuffle(buffer_size=3, seed=None)
for instance in dataset:
    print(instance)

tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)
tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([-0.63 -0.77 -0.33], shape=(3,), dtype=float32)
tf.Tensor([ 0.51 -0.05  0.26], shape=(3,), dtype=float32)
tf.Tensor([0.9  0.35 0.59], shape=(3,), dtype=float32)
tf.Tensor([ 0.45 -0.56  0.71], shape=(3,), dtype=float32)


In [13]:
# Using shuf
from pathlib import Path

s = "\n".join(",".join(f"m{line}{col}" for col in range(3)) for line in range(5))
print("Original data")
print(s)

path_file_txt = Path("tmp.txt")
with open(path_file_txt, "w") as fp:
    fp.write(s)

print("\nOption 1: with a file")
!shuf $path_file_txt

print("\nOption 2: using standard input")
s = s.replace("\n", " ")
!shuf -e $s

path_file_txt.unlink()

Original data
m00,m01,m02
m10,m11,m12
m20,m21,m22
m30,m31,m32
m40,m41,m42

Option 1: with a file
m40,m41,m42
m00,m01,m02
m10,m11,m12
m20,m21,m22
m30,m31,m32

Option 2: using standard input
m40,m41,m42
m20,m21,m22
m10,m11,m12
m30,m31,m32
m00,m01,m02


##### Interleaving Lines from Multiple Files

In [14]:
from sklearn.datasets import fetch_california_housing

dataset = fetch_california_housing()
print(dataset.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

    :Number of Instances: 20640

    :Number of Attributes: 8 numeric, predictive attributes and the target

    :Attribute Information:
        - MedInc        median income in block group
        - HouseAge      median house age in block group
        - AveRooms      average number of rooms per household
        - AveBedrms     average number of bedrooms per household
        - Population    block group population
        - AveOccup      average number of household members
        - Latitude      block group latitude
        - Longitude     block group longitude

    :Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived

In [15]:
dataset.feature_names

['MedInc',
 'HouseAge',
 'AveRooms',
 'AveBedrms',
 'Population',
 'AveOccup',
 'Latitude',
 'Longitude']

In [16]:
dataset.target_names

['MedHouseVal']

In [17]:
# Train test split
from sklearn.model_selection import train_test_split

X = dataset.data
y = dataset.target

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=0.2, random_state=42, shuffle=True)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)

X_train: (13209, 8)
X_valid: (3303, 8)
X_test: (4128, 8)


In [18]:
# Split into CSV files
import numpy as np
from collections import OrderedDict

data = OrderedDict([("train", (X_train, y_train)), ("valid", (X_valid, y_valid)), ("test", (X_test, y_test))])
n_files = OrderedDict([("train", 5), ("valid", 3), ("test", 3)])

data_groups = list(data.keys())
data_slices = OrderedDict([(data_group, np.array_split(np.c_[data[data_group][0], data[data_group][1]], n_file)) for data_group, n_file in n_files.items()])

for data_group in data_groups:
    print(f"Data group: {data_group}")
    for i_slice, data_slice in enumerate(data_slices[data_group]):
        print(f"  slice {i_slice}: {data_slice.shape}")

Data group: train
  slice 0: (2642, 9)
  slice 1: (2642, 9)
  slice 2: (2642, 9)
  slice 3: (2642, 9)
  slice 4: (2641, 9)
Data group: valid
  slice 0: (1101, 9)
  slice 1: (1101, 9)
  slice 2: (1101, 9)
Data group: test
  slice 0: (1376, 9)
  slice 1: (1376, 9)
  slice 2: (1376, 9)


In [19]:
# Save files
import pandas as pd
from pathlib import Path

col_names = dataset.feature_names + dataset.target_names

dir_data = Path("data")
dir_data.mkdir(exist_ok=True)
paths_data_files_csv = OrderedDict()
for data_group, data_slice_group in data_slices.items():
    paths_data_files_csv[data_group] = []
    for i_slice, data_slice in enumerate(data_slice_group, start=1):
        path_data_file_csv = dir_data / f"x_{data_group}_{i_slice}.csv"
        paths_data_files_csv[data_group].append(str(path_data_file_csv))
        df_data_slice = pd.DataFrame(data=data_slice, columns=col_names)
        df_data_slice.to_csv(path_data_file_csv, index=False)

df_data_slice.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,3.9531,35.0,5.426891,1.035294,1879.0,3.157983,34.05,-118.13,2.684
1,3.2708,26.0,4.269802,1.066832,1130.0,2.797030,33.97,-118.35,1.281
2,5.0767,25.0,6.042065,0.969407,1514.0,2.894837,34.44,-119.89,2.712
3,3.8650,30.0,6.060669,1.058577,1508.0,3.154812,38.69,-121.39,0.884
4,2.0091,21.0,3.802966,1.021186,1114.0,2.360169,38.00,-121.32,1.015


In [20]:
# File path dataset
dataset_paths = tf.data.Dataset.list_files("data/x_train_*.csv")
for instance in dataset_paths:
    print(instance)

tf.Tensor(b'data/x_train_3.csv', shape=(), dtype=string)
tf.Tensor(b'data/x_train_2.csv', shape=(), dtype=string)
tf.Tensor(b'data/x_train_1.csv', shape=(), dtype=string)
tf.Tensor(b'data/x_train_5.csv', shape=(), dtype=string)
tf.Tensor(b'data/x_train_4.csv', shape=(), dtype=string)


In [21]:
n_readers = 3
dataset = dataset_paths.interleave(lambda path: tf.data.TextLineDataset(path).skip(1), cycle_length=n_readers, num_parallel_calls=tf.data.experimental.AUTOTUNE)
len(list(dataset))
for instance in dataset.take(5):
    print(instance)

tf.Tensor(b'4.7069,27.0,6.523255813953488,1.1162790697674418,873.0,3.383720930232558,38.0,-120.97,1.769', shape=(), dtype=string)
tf.Tensor(b'3.7419,52.0,5.121890547263682,1.0348258706467661,999.0,2.485074626865672,37.79,-122.46,5.00001', shape=(), dtype=string)
tf.Tensor(b'6.203,38.0,6.26431718061674,1.024229074889868,2263.0,2.4922907488986783,37.74,-122.45,3.468', shape=(), dtype=string)
tf.Tensor(b'3.875,15.0,5.058405682715075,1.0757695343330702,3359.0,2.6511444356748224,34.1,-117.87,1.733', shape=(), dtype=string)
tf.Tensor(b'2.5781,28.0,5.48062015503876,1.193798449612403,561.0,4.348837209302325,34.0,-117.69,1.116', shape=(), dtype=string)


### 13.1.3 Preprocessing the Data

In [22]:
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

print("X_mean:", X_mean)
print("X_std:", X_std)

X_mean: [ 3.86893364e+00  2.85672647e+01  5.42040408e+00  1.09433536e+00
  1.42691650e+03  3.02944025e+00  3.56468476e+01 -1.19583303e+02]
X_std: [1.88982423e+00 1.25889927e+01 2.11894782e+00 3.81057844e-01
 1.13717053e+03 6.85823367e+00 2.13376293e+00 2.00518876e+00]


In [23]:
tf.io.decode_csv(list(dataset.take(1))[0], record_defaults=[0.] * 8 + [tf.constant([], dtype=tf.float32)])

[<tf.Tensor: shape=(), dtype=float32, numpy=4.7069>,
 <tf.Tensor: shape=(), dtype=float32, numpy=27.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=6.523256>,
 <tf.Tensor: shape=(), dtype=float32, numpy=1.1162791>,
 <tf.Tensor: shape=(), dtype=float32, numpy=873.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=3.3837209>,
 <tf.Tensor: shape=(), dtype=float32, numpy=38.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=-120.97>,
 <tf.Tensor: shape=(), dtype=float32, numpy=1.769>]

In [24]:
from typing import Tuple

n_features = 8

# def preprocess(line: str) -> Tuple[tf.Tensor, tf.Tensor]:
#     default_values = [0.] * n_features + [tf.constant([], dtype=tf.float32)]
#     fields = tf.io.decode_csv(line, record_defaults=default_values)
#     x = tf.stack(fields[:-1])
#     y = tf.stack(fields[-1:])
#
#     return (x - X_mean) / X_std, y

def preprocess(line: str):
    default_values = [0.] * n_features + [tf.constant([], dtype=tf.float32)]
    fields = tf.io.decode_csv(line, record_defaults=default_values)
    x = tf.stack(fields[:-1])
    y = tf.stack(fields[-1:])

    return (x - X_mean) / X_std, y

### 13.1.4 Putting Everything Together

In [25]:
from typing import Optional, Sequence

def csv_to_dataset(paths_data_csv: Sequence[str], n_repeat: int = 1, n_readers: int = 3, shuffle_buffer_size: int = 10000, batch_size: int = 32) -> tf.data.Dataset:
    # Create dataset from files
    dataset = tf.data.Dataset.list_files(paths_data_csv, seed=42)

    # Interleave
    dataset = dataset.interleave(lambda path: tf.data.TextLineDataset(path).skip(1), cycle_length=n_readers, num_parallel_calls=tf.data.AUTOTUNE)

    # Shuffle
    dataset = dataset.shuffle(shuffle_buffer_size, seed=42)

    # Repeat
    dataset = dataset.repeat(n_repeat)

    # Preprocess
    dataset = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch
    dataset = dataset.batch(batch_size)

    # Prefetch
    return dataset.prefetch(1)

### 13.1.5 Prefetching

### 13.1.6 Using the Dataset with `tf.keras`

In [26]:
train_set = csv_to_dataset(paths_data_files_csv["train"])
valid_set = csv_to_dataset(paths_data_files_csv["valid"])
test_set = csv_to_dataset(paths_data_files_csv["test"])

train_set

<PrefetchDataset element_spec=(TensorSpec(shape=(None, 8), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))>

In [27]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(10, activation="selu", kernel_initializer="lecun_normal", input_shape=(8,)),
    tf.keras.layers.Dense(3, activation="selu", kernel_initializer="lecun_normal"),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer="rmsprop", loss="huber", metrics="mse")
model.summary()

ImportError: cannot import name 'dtensor' from 'tensorflow.compat.v2.experimental' (/home/yu/anaconda3/envs/tensorflow/lib/python3.10/site-packages/tensorflow/_api/v2/compat/v2/experimental/__init__.py)

In [ ]:
# TensorBoard
import time
from tensorflow.keras.callbacks import TensorBoard

def dir_log() -> str:
    return str(Path() / "tensorboard_logs" / time.strftime("run_%Y_%m_%d-%H_%M_%S"))

%load_ext tensorboard
%tensorboard --logdir=./tensorboard_logs --port=6006

In [ ]:
# Early stopping
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(patience=10, restore_best_weights=True)

In [ ]:
history = model.fit(train_set, epochs=100, callbacks=[TensorBoard(dir_log()), early_stopping], validation_data=valid_set)

In [ ]:
model.evaluate(test_set, return_dict=True)

In [ ]:
X_new_batches, y_new_batches, = list(zip(*[(X, y) for X, y in test_set.take(3)]))
X_new = tf.concat(X_new_batches, 0)
y_new = tf.concat(y_new_batches, 0)
y_pred = model.predict(X_new)
pd.DataFrame({"y_pred": y_pred.ravel(), "y_true": np.asarray(y_new).ravel()})

In [ ]:
# Custom training loop
@tf.function
def train(model: tf.keras.Model, optimizer: tf.optimizers.Optimizer, loss_fn: tf.losses.Loss, n_epochs: int, **kwargs) -> None:
    train_set = csv_to_dataset(paths_data_csv=paths_data_files_csv["train"], n_repeat=n_epochs, **kwargs)
    for X_batch, y_batch in train_set:
        with tf.GradientTape() as tape:
            y_pred = model(X_batch)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

# train(model, optimizer=tf.optimizers.RMSprop(), loss_fn=tf.losses.MeanSquaredError(), n_epochs=10)
#
# X_new_batches, y_new_batches, = list(zip(*[(X, y) for X, y in test_set.take(3)]))
# X_new = tf.concat(X_new_batches, 0)
# y_new = tf.concat(y_new_batches, 0)
# y_pred = model.predict(X_new)
# pd.DataFrame({"y_pred": y_pred.ravel(), "y_true": np.asarray(y_new).ravel()})

## 13.2 The TFRecord Format

##### TFRecord

In [ ]:
path_file_tfrecord = "my_data.tfrecord"

options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter(path_file_tfrecord, options) as fp:
    fp.write(b"This is the first record")
    fp.write(b"This is the second record")

dataset = tf.data.TFRecordDataset([path_file_tfrecord], compression_type="GZIP")
for item in dataset:
    print(item)

##### Protocol Buffs

In [ ]:
from tensorflow.train import BytesList, FloatList, Int64List, Feature, Features, Example
from typing import Any, Dict, Union

path_contacts_tfrecord = "my_contacts.tfrecord"
contacts = {
    "name": [b"Alice", b"Bob", b"Cindy"],
    "id": [1, 2, 3],
    "emails": [[b"alice@gmail.com", b"alice@outlook.com"], [b"bob@web.de"], [b"c@163.com", b"cindy@uu.nl"]]
}

def dict_to_tfrecord(path_tfrecord: Union[str, Path], d: Dict[str, Any]) -> None:
    df = pd.DataFrame(d)
    with tf.io.TFRecordWriter(path_tfrecord) as fp:
        for record in df.itertuples():
            example = Example(features=Features(feature={
                "name": Feature(bytes_list=BytesList(value=[record.name])),
                "id": Feature(int64_list=Int64List(value=[record.id])),
                "emails": Feature(bytes_list=BytesList(value=[*record.emails])),
            }))
            fp.write(example.SerializeToString())

dict_to_tfrecord(path_contacts_tfrecord, contacts)

feature_description = {
    "name": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails": tf.io.VarLenFeature(tf.string),
}

for serialized_example in tf.data.TFRecordDataset([path_contacts_tfrecord]):
    parsed_example = tf.io.parse_single_example(serialized_example, feature_description)
    print(" - Name: " + str(parsed_example["name"]))
    print(" - ID: " + str(parsed_example["id"]))
    print(" - Emails: " + str(parsed_example["emails"].values) + "\n")

dataset = tf.data.TFRecordDataset([path_contacts_tfrecord])
for serialized_examples in dataset.batch(2):
    parsed_examples = tf.io.parse_example(serialized_examples, feature_description)
    print(parsed_examples)

## 13.3 Preprocessing the Input Features

In [ ]:
class Standardization(tf.keras.layers.Layer):
    def adapt(self, data_samples: tf.Tensor) -> None:
        self.means_ = tf.reduce_mean(tf.constant(data_samples, dtype=tf.float32), axis=0, keepdims=True)
        self.stds_ = tf.math.reduce_std(tf.constant(data_samples, dtype=tf.float32), axis=0, keepdims=True)
    def call(self, inputs, **kwargs) -> tf.Tensor:
        return (inputs - self.means_) / (self.stds_ + tf.keras.backend.epsilon())

standardization = Standardization()
standardization.adapt(X_train)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(8,)),
    standardization,
    tf.keras.layers.Dense(10, activation="selu", kernel_initializer="lecun_normal"),
    tf.keras.layers.Dense(3, activation="selu", kernel_initializer="lecun_normal"),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer="rmsprop", loss="huber", metrics="mse")
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=100, callbacks=[TensorBoard(dir_log()), early_stopping], validation_data=(X_valid, y_valid))

### 13.3.1 Encoding Categorical Features Using One-Hot Vectors

In [ ]:
import tarfile
from urllib.request import urlretrieve
from yarl import URL

url_data_csv = URL("https://github.com/ageron/handson-ml/raw/master/datasets/housing/housing.tgz")
dir_data = Path("data", "california_housing")
path_data_tgz = dir_data / "housing.tgz"
path_data_csv = path_data_tgz.with_suffix(".csv")

dir_data.mkdir(exist_ok=True)
if not path_data_tgz.is_file():
    urlretrieve(str(url_data_csv), str(path_data_tgz))
if not path_data_csv.is_file():
    with tarfile.open(path_data_tgz) as fp:
        fp.extractall(dir_data)

list(path_data_csv.parent.iterdir())

In [ ]:
df_data = pd.read_csv(path_data_csv)
df_data.info()

In [ ]:
df_data.describe()

In [ ]:
df_data.head()

In [ ]:
# Data cleansing
from sklearn.impute import SimpleImputer

col_cat = "ocean_proximity"
cols_num = df_data.columns[df_data.columns != col_cat]
df_data_num = df_data[cols_num]

data_num = SimpleImputer(strategy="median").fit_transform(df_data_num).astype(np.float32)
data_cat = df_data[col_cat].values

pd.DataFrame(data_num, columns=cols_num).info()

In [ ]:
# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(np.c_[data_num[:, :8], data_cat], data_num[:, 8], test_size=0.20, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Standardization layer
import tensorflow as tf

standardization.adapt(X_train[:, :-1])

In [ ]:
# One-hot encoding layer
class OneHot(tf.keras.layers.Layer):
    def adapt(self, data_sample: np.ndarray, n_oov: int = 2) -> None:
        self.vocab_ = tf.unique(data_sample.ravel()).y
        self.n_oov_ = n_oov
        indices = tf.range(self.vocab_.shape[0], dtype=tf.int64)
        table_init = tf.lookup.KeyValueTensorInitializer(self.vocab_, indices)
        self.table_ = tf.lookup.StaticVocabularyTable(table_init, n_oov)

    def call(self, inputs, **kwargs):
        indices = self.table_.lookup(tf.reshape(inputs, [-1]))
        return tf.one_hot(indices, depth=len(self.vocab_) + self.n_oov_)

one_hot = OneHot()
one_hot.adapt(X_train[:, -1])
one_hot.vocab_

In [ ]:
one_hot(["NEAR BAY", "DESERT", "INLAND", "INLAND"])

In [ ]:
df_data[:8]

### 13.3.2 Encoding Categorical Features Using Embeddings

In [ ]:
vocab = np.unique(X_train[:, -1])
n_words = len(vocab)
n_oov = 2
n_dims_embedding = 2
embedding_init = tf.random.uniform([n_words + n_oov, n_dims_embedding])
embedding_matrix = tf.Variable(embedding_init)

print("Vocabulary:")
print(vocab)
print("\nEmbedding matrix:")
print(embedding_matrix)

In [ ]:
instances = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND"])

indices = tf.range(n_words, dtype=tf.int64)
table_init = tf.lookup.KeyValueTensorInitializer(vocab, indices)
table = tf.lookup.StaticVocabularyTable(table_init, n_oov)
instance_indices = table.lookup(instances)
print(instance_indices)
tf.nn.embedding_lookup(embedding_matrix, instance_indices)

In [ ]:
inputs = {
  'numerical': tf.keras.Input(shape=(8,), dtype=tf.float32),
  'categorical': tf.keras.Input(shape=()),
}

inputs_num = tf.keras.layers.Input(shape=(8,))
hidden_num = standardization(inputs_num)

inputs_cat = tf.keras.layers.Input(shape=[], dtype=tf.string)
hidden_cat = tf.keras.layers.Lambda(lambda cats: table.lookup(cats))(inputs_cat)
hidden_cat = tf.keras.layers.Embedding(input_dim=n_words, output_dim=n_dims_embedding)(hidden_cat)

hidden_com = tf.keras.layers.concatenate([hidden_num, hidden_cat])
hidden_com = tf.keras.layers.Dense(10, activation="selu", kernel_initializer="lecun_normal")(hidden_com)
hidden_com = tf.keras.layers.Dense(3, activation="selu", kernel_initializer="lecun_normal")(hidden_com)
output = tf.keras.layers.Dense(1)(hidden_com)

model = tf.keras.Model(inputs=[inputs_num, inputs_cat], outputs=[output])

model.compile(optimizer="rmsprop", loss="huber", metrics="mse")
model.summary()

In [ ]:
def df_to_dataset(
    df: pd.DataFrame,
    cols_num: Sequence[str],
    cols_cat: Sequence[str],
    col_target: str,
    shuffle : bool =True,
    batch_size : int =32
) -> tf.data.Dataset:
    df = df.copy()
    labels = df[col_target]
    feature_num = df[cols_num].astype(float)
    feature_cat = df[cols_cat].astype(str)
    ds = tf.data.Dataset.from_tensor_slices(((feature_num, feature_cat), labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(inputs))
    ds = ds.batch(batch_size)
    ds = ds.prefetch(batch_size)
    return ds

In [ ]:
col_cat = "ocean_proximity"
col_target = "median_house_value"
cols_num = [col for col in df_data if col not in [col_cat, col_target]]

print(cols_num)
print(col_cat)
print(col_target)

In [ ]:
df_data_imp = df_data.copy()
df_data_imp[cols_num] = SimpleImputer(strategy="median").fit_transform(df_data[cols_num])
df_data_imp.head()

In [ ]:
df_data_imp.info()

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df_data_imp, test_size=0.1, shuffle=False)
df_train, df_valid = train_test_split(df_train, test_size=0.2, shuffle=False)
dss = {ds_name: df_to_dataset(df=df_ds, cols_num=cols_num, cols_cat=[col_cat], col_target=col_target) for ds_name, df_ds in zip(["train", "valid", "test"], [df_train, df_valid, df_test])}

print("Train: ", len(df_train))
print("Valid: ", len(df_valid))
print("Test: ", len(df_test))

In [ ]:
history = model.fit(dss["train"], epochs=1000, callbacks=[TensorBoard(dir_log()), early_stopping], validation_data=dss["valid"])

In [ ]:
model.predict(dss["test"].take(1))

In [ ]:
for ins in dss["test"].take(1):
    print(ins)

In [ ]:
# Tutorial: https://www.tensorflow.org/tutorials/structured_data/feature_columns
# TODO: standarize output values

### 13.3.3 Keras Preprocessing Layers

In [ ]:
normalization = tf.keras.layers.Nor

## 13.4 TF Transform

## 13.5 The TensorFlow Datasets (TFDS) Project